In [27]:
df=pd.read_csv("/content/inventory_data.csv")


In [3]:
import pandas as pd


Inventory Analysis

In [6]:
#1)Which medical items contribute to the highest inventory value?
#inventory value=currentstock*unitcost
df['inventory_value'] = df['Current_Stock'] * df['Unit_Cost']
#Answer is Ventilator which is a consumable

In [7]:
df.sort_values("inventory_value",ascending=False)

,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID,inventory_value
156,2025-03-06,108,Consumable,Ventilator,4864,771,2669,18329.75,87,9,V003,89155904.00
87,2024-12-27,105,Consumable,IV Drip,4485,478,5480,19401.95,484,2,V002,87017745.75
300,2025-07-28,100,Equipment,X-ray Machine,4518,675,4916,19140.66,430,13,V002,86477501.88
102,2025-01-11,102,Consumable,Gloves,4377,595,5296,19127.81,344,5,V003,83722424.37
361,2025-09-27,108,Equipment,Surgical Mask,4974,942,1360,16517.32,373,24,V002,82157149.68
...,...,...,...,...,...,...,...,...,...,...,...,...
408,2025-11-13,102,Consumable,X-ray Machine,300,591,2399,559.59,183,23,V002,167877.00
412,2025-11-17,101,Equipment,Surgical Mask,189,282,5654,879.04,245,8,V001,166138.56
127,2025-02-05,100,Consumable,Surgical Mask,69,440,1359,1353.10,410,14,V002,93363.90
311,2025-08-08,100,Equipment,Ventilator,248,293,1024,198.75,470,8,V001,49290.00


In [8]:
#2)Which items show low movement but high holding cost?
#item which shoul have low movement means avg usage per day and high value
#Answer is x-Ray machine which is consumable

In [9]:
df.sort_values(["Avg_Usage_Per_Day","inventory_value"],ascending=[True,False])

,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID,inventory_value
366,2025-10-02,106,Consumable,X-ray Machine,860,400,855,18760.33,2,11,V003,16133883.80
357,2025-09-23,100,Consumable,Surgical Mask,1196,367,4168,8853.64,2,26,V003,10588953.44
89,2024-12-29,101,Consumable,Ventilator,1434,585,5992,18268.70,4,21,V001,26197315.80
362,2025-09-28,104,Consumable,Gloves,865,757,728,15373.96,4,19,V003,13298475.40
433,2025-12-08,101,Equipment,X-ray Machine,1562,169,2262,5321.71,6,24,V001,8312511.02
...,...,...,...,...,...,...,...,...,...,...,...,...
314,2025-08-11,104,Consumable,X-ray Machine,4223,967,2997,12101.98,494,21,V002,51106661.54
136,2025-02-14,108,Equipment,Ventilator,1466,199,3397,14687.35,496,18,V001,21531655.10
193,2025-04-12,108,Consumable,IV Drip,3389,421,5580,15278.79,498,29,V003,51779819.31
158,2025-03-08,109,Equipment,Surgical Mask,1078,784,3744,6971.80,499,18,V001,7515600.40


In [10]:
#3)Which departments experience frequent stockouts?
#frequent stockout would be current stock<min required
#Answer is "Consumable Item"

In [11]:
df['Stockout_Flag']=df['Current_Stock']<df['Min_Required']

In [12]:
df

,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID,inventory_value,Stockout_Flag
0,2024-10-01,105,Consumable,Ventilator,1542,264,1018,4467.55,108,17,V001,6888962.10,False
1,2024-10-02,100,Equipment,Ventilator,2487,656,3556,5832.29,55,12,V001,14504905.23,False
2,2024-10-03,103,Equipment,Surgical Mask,2371,384,5562,16062.98,470,6,V001,38085325.58,False
3,2024-10-04,103,Consumable,Surgical Mask,2038,438,1131,744.10,207,15,V002,1516475.80,False
4,2024-10-05,107,Equipment,IV Drip,2410,338,1013,15426.53,158,12,V003,37177937.30,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2026-02-08,108,Consumable,Ventilator,4072,776,3496,15934.20,257,6,V002,64884062.40,False
496,2026-02-09,100,Equipment,X-ray Machine,997,239,2186,5863.88,108,16,V002,5846288.36,False
497,2026-02-10,107,Equipment,Surgical Mask,4193,304,4213,90.30,419,27,V001,378627.90,False
498,2026-02-11,106,Consumable,IV Drip,822,186,2385,5284.45,382,6,V003,4343817.90,False


In [13]:
df.groupby('Item_Type')['Stockout_Flag'].sum()

,Stockout_Flag
Item_Type,
Consumable,27
Equipment,16


In [19]:
#4)Are reorder levels aligned with actual consumption?
#Reorder Level=Avg_Usage_Per_Day * Restock_Lead_Time
#aligned with actual cosumption=current stock /reorderlevel
#Answer would be
# Consumable	Understocked	129
#Overstocked	102
#Well Aligned	35
#Equipment	Understocked	131
#Overstocked	73
#




df["Reorder_level"]=df["Avg_Usage_Per_Day"]*df['Restock_Lead_Time']
df["Reorder_Alignment_Ratio"]=df['Current_Stock']/df['Reorder_level']


In [20]:
df['Reorder_Category'] = 'Well Aligned'

df.loc[df['Reorder_Alignment_Ratio'] < 0.8, 'Reorder_Category'] = 'Understocked'
df.loc[df['Reorder_Alignment_Ratio'] > 1.2, 'Reorder_Category'] = 'Overstocked'


In [21]:
df.groupby("Item_Type")["Reorder_Category"].value_counts()

Item_Type   Reorder_Category
Consumable  Understocked        129
            Overstocked         102
            Well Aligned         35
Equipment   Understocked        131
            Overstocked          73
            Well Aligned         30
Name: count, dtype: int64

In [22]:
#5)- Which categories have the lowest inventory turnover?
#inv turn over=gives how well a inventory is used
#inventory turnover=avg usage per day/current stock
#Answer is Consumables
df['Inventory_Turn_over_Ratio']=df['Avg_Usage_Per_Day']/df['Current_Stock']

In [24]:
df.sort_values("Inventory_Turn_over_Ratio")

,Date,Item_ID,Item_Type,Item_Name,Current_Stock,Min_Required,Max_Capacity,Unit_Cost,Avg_Usage_Per_Day,Restock_Lead_Time,Vendor_ID,inventory_value,Stockout_Flag,Reorder_Category,Reorder_level,Reorder_Alignment_Ratio,Inventory_Turn_over_Ratio
357,2025-09-23,100,Consumable,Surgical Mask,1196,367,4168,8853.64,2,26,V003,10588953.44,False,Overstocked,52,23.000000,0.001672
27,2024-10-28,103,Consumable,X-ray Machine,4289,21,1823,9547.31,9,21,V001,40948412.59,False,Overstocked,189,22.693122,0.002098
245,2025-06-03,107,Consumable,IV Drip,4140,60,3197,14127.56,9,7,V002,58488098.40,False,Overstocked,63,65.714286,0.002174
366,2025-10-02,106,Consumable,X-ray Machine,860,400,855,18760.33,2,11,V003,16133883.80,False,Overstocked,22,39.090909,0.002326
472,2026-01-16,109,Equipment,Ventilator,3055,741,2386,12691.40,8,28,V001,38772227.00,False,Overstocked,224,13.638393,0.002619
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
461,2026-01-05,107,Consumable,Surgical Mask,160,427,5480,16158.21,481,17,V003,2585313.60,True,Understocked,8177,0.019567,3.006250
113,2025-01-22,103,Equipment,X-ray Machine,139,58,5874,4233.14,420,19,V002,588406.46,False,Understocked,7980,0.017419,3.021583
293,2025-07-21,101,Consumable,Gloves,156,403,2447,15351.22,486,24,V001,2394790.32,True,Understocked,11664,0.013374,3.115385
391,2025-10-27,105,Consumable,X-ray Machine,93,61,5214,2892.86,348,2,V002,269035.98,False,Understocked,696,0.133621,3.741935


In [26]:
df.groupby("Item_Type")["Inventory_Turn_over_Ratio"].value_counts()

Item_Type   Inventory_Turn_over_Ratio
Consumable  0.001672                     1
            0.002098                     1
            0.002174                     1
            0.002326                     1
            0.002789                     1
                                        ..
Equipment   2.008772                     1
            2.089431                     1
            2.350000                     1
            2.480000                     1
            3.021583                     1
Name: count, Length: 500, dtype: int64

In [29]:
#6)How frequently does emergency procurement occur?
df['Emergency_Procurement_Flag'] = (
    df['Current_Stock'] < df['Min_Required']
).astype(int)


In [31]:
df.groupby('Item_Type')['Emergency_Procurement_Flag'].mean().sort_values(ascending=False) * 100


,Emergency_Procurement_Flag
Item_Type,
Consumable,10.150376
Equipment,6.837607


In [32]:
#7) Is safety stock adequate for critical items?
# critical item which is inadequate safety stock number is 17

In [33]:
df['LT_Demand'] = df['Avg_Usage_Per_Day'] * df['Restock_Lead_Time']
df['Safety_Stock'] = df['Current_Stock'] - df['Min_Required']
df['Safety_Stock_Ratio'] = df['Safety_Stock'] / df['LT_Demand']
df['Safety_Stock_Status'] = 'Adequate'

df.loc[df['Safety_Stock_Ratio'] < 0, 'Safety_Stock_Status'] = 'No Safety Stock'
df.loc[df['Safety_Stock_Ratio'] < 0.5, 'Safety_Stock_Status'] = 'Inadequate'
df.loc[df['Safety_Stock_Ratio'] > 1, 'Safety_Stock_Status'] = 'Excess'


In [36]:
usage_threshold = df['Avg_Usage_Per_Day'].quantile(0.75)
df['Critical_Flag'] = df['Avg_Usage_Per_Day'] >= usage_threshold



In [42]:
df.loc[(df["Critical_Flag"]==True)&(df['Safety_Stock']=="Inadequate")].shape

(0, 17)